<a href="https://colab.research.google.com/github/nirmit234/-Machine-Learning-Projects/blob/main/ift_6390_nirmit_mukeshkumar_rathod_final_kaggle_python_code_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
  from google.colab import files
  import zipfile, glob
  files.upload()                                   # choose the-llm-safety-mandate zip
  zipfile.ZipFile(glob.glob("*.zip")[0]).extractall("/content/data")

Saving the-llm-safety-mandate.zip to the-llm-safety-mandate.zip


In [ ]:
import glob, os, warnings
import numpy as np, pandas as pd
# the pipeline steps
  #finds the 7 clusters
  # model 1: logistic regression
  # model 2: boosted trees
  # model 3: random forest
  # averages the three models
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler                # puts features on the same scale for the logistic model

warnings.filterwarnings("ignore")

# locate files
roots = ["/kaggle/input", "/content", "."]                      # Kaggle, Colab, current folder
hits = [p for r in roots if os.path.isdir(r)
        for p in glob.glob(os.path.join(r, "**", "train.csv"), recursive=True)]
if not hits:
    raise FileNotFoundError("train.csv not found. Upload and unzip the competition data first.")
DATA = os.path.dirname(hits[0])                                 # folder that contains train.csv
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
print("Using data from:", DATA)

train = pd.read_csv(f"{DATA}/train.csv")
test = pd.read_csv(f"{DATA}/test.csv")
sample = pd.read_csv(f"{DATA}/sample_submission.csv")

# The five "named" features. They carry only weak, extra information, so we keep them as they are.
COUNTS = ["modal_conditionnel", "mise_a_distance", "imperatif_debut", "n_obligations", "part_majuscules"]

# Cluster the (v1, v2) plane into 7 groups.
  # 7 was chosen because it has the best silhouette score (0.81) of the values 3..10 from which I tried.
  # n_init=20 restarts KMeans 20 times and keeps the best, so the clusters are stable.
  # random_state=0 makes the result identical every run.
  # We fit on train AND test together.
km = KMeans(7, n_init=20, random_state=0).fit(pd.concat([train[["v1", "v2"]], test[["v1", "v2"]]]).values)

# apply the same transformation to both files
for d in (train, test):
    d["c"] = km.predict(d[["v1", "v2"]].values)                  # c   = cluster number 0..6
    d["s3"] = (d.v3 > 0).astype(int)                             # s3  = 1 if v3 is in its positive blob
    d["s4"] = (d.v4 > 0).astype(int)                             # s4  = 1 if v4 is in its positive blob
    d["s34"] = d.s3 * d.s4                                       # s34 = 1 only when BOTH are positive (the "flip" switch)
y = train["refus"].values                                        # the answers
BASE = COUNTS + ["c", "s3", "s4", "s34"]                         # columns handed to the pipelines below

#  PIPELINE THAT BUILDS THE XOR FEATURE
class Rule(BaseEstimator, TransformerMixin):
  """
    fit() : looks at the training labels and decide which clusters are 'high refusal'.
    transform(): add  g=1 if the row's cluster is high-refusal, else 0
                      rule  = g XOR s34      <- the feature that carries the pattern

    XOR rules: 1 XOR 0 = 1,  0 XOR 1 = 1,  1 XOR 1 = 0,  0 XOR 0 = 0.
    Example: a cluster-0 row (high refusal, g=1)
        if v3, v4 not both positive then -> s34=0 -> rule = 1 XOR 0 = 1  -> predict refusal
        if v3 and v4 both positive then -> s34=1 -> rule = 1 XOR 1 = 0  -> the flip -> predict no refusal
    """
  def fit(self, X, y):
      # average refusal rate per cluster; a cluster counts as 'high refusal' if more than 45% of its rows refuse

      self.grp_ = (pd.Series(np.asarray(y)).groupby(X.c.values).mean() > 0.45).astype(int).to_dict()
      # for example {0:1, 1:1, 2:0, 3:0, 4:1, 5:0, 6:0}
      return self

  def transform(self, X):
      X = X.copy()
      X["g"] = X.c.map(self.grp_).fillna(0).astype(int)   # look up each row's cluster group
      X["rule"] = X.g ^ X.s34                             # ^ is XOR in Python
      return X


class Cols(BaseEstimator, TransformerMixin):
    """Pipeline that simply keeps a chosen list of columns (so each model sees only what we intend)."""

    def __init__(self, cols): self.cols = cols
    def fit(self, X, y=None): return self
    def transform(self, X): return X[self.cols].values


def pipe(model, cols):
    """Chain: build rule feature -> select columns -> model.
    Because Rule() is INSIDE the pipeline, during cross-validation it is re-learned from the training
    part of each fold only, so the validation score is not inflated by peeking at held-out labels."""

    return Pipeline([("rule", Rule()), ("cols", Cols(cols)), ("m", model)])


# THE All THREE MODELS AND THEIR VOTES


# five counts + the XOR feature
SMALL = COUNTS + ["rule"]
# same, plus the pieces the rule is built from
WIDE = COUNTS + ["rule", "g", "s3", "s4", "s34", "c"]
model = VotingClassifier([
    # Model 1 - logistic regression: simple and smooth. C=0.3 = fairly strong regularisation (resists overfitting).
    ("lr", pipe(Pipeline([("sc", StandardScaler()), ("m", LogisticRegression(C=0.3, max_iter=2000))]), SMALL)),
    # Model 2 - gradient boosting, deliberately shallow (max_depth=2) with a small learning rate and
    # large minimum leaf size: it can pick up gentle non-linear effects of the counts without memorising noise.
    ("hgb", pipe(HistGradientBoostingClassifier(max_depth=2, learning_rate=0.03, max_iter=150, min_samples_leaf=40,
                                                l2_regularization=5, random_state=0), SMALL)),
    # Model 3 - random forest: 500 trees, each leaf needs >= 15 rows, each split looks at half the features.
    # Gets the WIDE set so it can discover interactions on its own as a cross-check on the rule.
    ("rf", pipe(RandomForestClassifier(500, min_samples_leaf=15, max_features=0.5, random_state=0, n_jobs=1), WIDE)),
], voting="soft")
# soft = average the three predicted probabilities (instead of counting 0/1 votes)

# VALIDATE, TRAIN, PREDICTXtr, Xte = train[BASE], test[BASE]
cv = cross_val_score(model, Xtr, y, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="accuracy")
print(f"5-fold CV accuracy: {cv.mean():.4f} (+-{cv.std():.4f})")

# final training on ALL 1800 labelled rows
model.fit(Xtr, y)
# probability of refus=1 for each test row
proba = (model.predict_proba(Xte)[:, 1] > 0.5).astype(int)
# more likely refusal than not -> 1, else 0
pred  = (proba > 0.5).astype(int)

# test the data
sub = pd.DataFrame({"id": test["id"].values, "refus": pred})


# checking if the columns, row count or id ORDER differ from sample_submission.csv.
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample) and (sub.id.values == sample.id.values).all()

# index=False: no extra row-number column
os.makedirs(OUT, exist_ok=True)
sub.to_csv(f"{OUT}/submission.csv", index=False)
print(f"Saved {OUT}/submission.csv  rows={len(sub)}  predicted refusals={pred.mean():.1%}")

Using data from: /content/data
5-fold CV accuracy: 0.8911 (+-0.0123)
Saved ./submission.csv  rows=1198  predicted refusals=48.8%


In [ ]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>